#**Part 4 — GenAI: turn call-centre notes into structured reasons, then recommend**

##Task 4.1 — One validated call

In [ ]:
!pip install -q -U google-genai pydantic

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.63.0 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.6.1 which is incompatible.


In [ ]:
import json
import pandas as pd

In [ ]:
from google import genai
from google.genai import types

In [ ]:
API_KEY = input("Paste your Gemini API key:")
client = genai.Client(api_key=API_KEY)

In [ ]:
MODEL_NAME = "gemini-3.6-flash"

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required":[
        "root_cause", "confidence", "summary", "recommended_action"
    ]
}

SYSTEM_TEXT = """
You analyze brief patient follow-up notes after missed clinic appointments.
Use only details supported by the note. Do not invent facts or diagnose.
Return the requested structured result.
"""

In [ ]:
def analyse_note(note_text):
  analyse_note.last_raw_response = ""

  prompt = f"""
Context: CareLoop Clinics is reviewing short call-centre notes about missed visits.

Role: You are a careful data analyst extracting a likely reason from one note.

Action: Classify the main reason the patient missed the appointment. If the note
is unclear or does not support another category, use "other".
Summarize the note in at most 25 words and recommend a practical follow-up action.

Allowed root_cause values:
forgot_or_no_reminder
distance_or_transparent
cost_or_fee
illness_or_emergency
work_or_schedule
other

Tone: Neutral, concise, and factual.

Format: Return exactly these JSON fields:
root_cause, confidence, summary, recommended_action.

Note:
{note_text}
"""

  response = client.models.generate_content(
      model=MODEL_NAME,
      contents=prompt,
      config=types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="low")
      )
  )

  #Save the exact text before attempting JSON parsing.
  raw_response = response.text or ""
  analyse_note.last_raw_response = raw_response

  if not raw_response.strip():
      raise ValueError("The model returned an empty response.")

  data = json.loads(raw_response)

  if not isinstance(data, dict):
    raise ValueError("The model response was valid JSON, but not an object.")

  return data

analyse_note.last_raw_response = ""

In [ ]:
notes = pd.read_csv("/content/followup_notes.csv")
note_n001 = notes.loc[notes["note_id"] == "N001", "note_text"].iloc[0]
result_n001 = analyse_note(note_n001)
print(result_n001)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'Patient missed the appointment because she received no reminder message and forgot. She requested to rebook for next week.', 'recommended_action': "Reschedule the patient's appointment for next week and verify that reminder notifications are properly configured."}


##Task 4.2 — Validate and process the 10 notes

In [ ]:
from pydantic import BaseModel, Field, ValidationError
import time

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

ALLOWED_ROOT_CAUSES = {
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
}

RESPONSE_FIELDS = [
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
]


OUTPUT_COLUMNS = [
    "note_id",
    "appointment_id",
    "root_cause",
    "confidence",
    "summary",
    "recommended_action",
    "validation_status",
    "raw_response"
]

In [ ]:
def is_blank(note_text):
  #Check for NaN before converting to string.
  return pd.isna(note_text) or str(note_text).strip() == ""


In [ ]:
notes_df = pd.read_csv("/content/followup_notes.csv")

In [ ]:
notes_first_10 = notes_df.head(10)

In [ ]:
results = []
api_calls_4_2 = 0

In [ ]:
for _, source_row in notes_first_10.iterrows():
  note_id = str(source_row["note_id"])
  appointment_id = source_row["appointment_id"]
  note_text = source_row["note_text"]

  #Start with IDs and empty response fields.
  record = {
      "note_id": note_id,
      "appointment_id": appointment_id,
      "root_cause": "",
      "confidence": "",
      "summary": "",
      "recommended_action": "",
      "validation_status": "",
      "raw_response": "",
  }

  if is_blank(note_text):
    record.update({
        "root_cause": "other",
        "confidence": 0.0,
        "summary": "No note recorded",
        "recommended_action": "Re-call patient",
        "validation_status": "skipped_blank",
        "raw_response": "",
    })
    results.append(record)
    print(f"{note_id}: blank note skipped; no API call.")
    continue

  note_text = str(note_text).strip()

  #Pause between nonblank API attempts.
  if api_calls_4_2 > 0:
    time.sleep(13)
  api_calls_4_2 += 1

  try:
    data = analyse_note(note_text)
    raw_response = analyse_note.last_raw_response

    # Copy this immediately after the call. It is separate from `data`,
    # so the returned dictionary keeps only its four analysis fields.
    record["raw_response"] = getattr(
        analyse_note, "last_raw_response", ""
    )

  except(json.JSONDecodeError, ValueError, TypeError) as exc:
    #The API replied, but the reply could not be parsed as usable JSON.
    record["raw_response"] = getattr(
        analyse_note, "last_raw_response", ""
    )
    record["validation_status"] = "invalid"
    print(f"{note_id}: invalid model reply: {exc}")
    results.append(record)
    continue

  except Exception as exc:
    #Keep API/network errors separate from invalid model replies.
    record["raw_response"] = getattr(
        analyse_note, "last_raw_response", ""
    )
    record["validation_status"] = "api_error"
    print(f"{note_id}: API/network failure: {exc}")
    results.append(record)
    continue

  if not isinstance(data, dict):
    record["validation_status"] = "invalid"
    print(f"{note_id}: invalid model reply; expected a JSON object.")
    results.append(record)
    continue

  #Preserve fields from the reply before validating it.
  #Missing fields remain empty; invalid values are not repaired.
  for field in RESPONSE_FIELDS:
    record[field] = data.get(field, "")

  try:
    insight = FollowupInsight(**data)
  except ValidationError as exc:
    record["validation_status"] = "invalid"
    print(f"{note_id}: Pydantic validation failed:\n{exc}")
  else:
    if insight.root_cause not in ALLOWED_ROOT_CAUSES:
      record["validation_status"] = "invalid"
      print(
          f"{note_id}: invalid root cause "
          f"{insight.root_cause!r}; not in the allowed list."
      )
    else:
      record["validation_status"] = "valid"
      print(f"{note_id}: valid.")
  results.append(record)

#Save exactly the required eight columns and all ten note rows.
results_df = pd.DataFrame(results, columns=OUTPUT_COLUMNS)
results_df.to_csv("notes_analysed.csv", index=False)

print(f"\nNotes processed: {len(results_df)}")
print(f"API attempts in Task 4.2: {api_calls_4_2}")
print("Status counts:")
print(results_df["validation_status"].value_counts(dropna=False))
print("\nSaved notes_analysed.csv")
display(results_df)




N001: valid.
N002: invalid root cause 'distance_or_transparent'; not in the allowed list.
N003: valid.
N004: blank note skipped; no API call.
N005: valid.
N006: valid.
N007: valid.
N008: invalid root cause 'distance_or_transparent'; not in the allowed list.
N009: valid.
N010: valid.

Notes processed: 10
API attempts in Task 4.2: 9
Status counts:
validation_status
valid            7
invalid          2
skipped_blank    1
Name: count, dtype: int64

Saved notes_analysed.csv


,note_id,appointment_id,root_cause,confidence,summary,recommended_action,validation_status,raw_response
0,N001,A0872,forgot_or_no_reminder,0.95,Patient forgot the appointment because she rec...,Reschedule the patient's appointment for next ...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
1,N002,A0897,distance_or_transparent,0.95,Patient missed appointment because they live f...,Explore telehealth options or assist with sche...,invalid,"{""root_cause"":""distance_or_transparent"",""confi..."
2,N003,A0526,cost_or_fee,0.95,Patient skipped the appointment due to high co...,Provide information regarding available discou...,valid,"{""root_cause"":""cost_or_fee"",""confidence"":0.95,..."
3,N004,A0926,other,0.00,No note recorded,Re-call patient,skipped_blank,
4,N005,A0534,illness_or_emergency,0.98,Patient missed the appointment due to a high f...,Follow up with the patient in a few days to ch...,valid,"{\n ""root_cause"": ""illness_or_emergency"",\n ..."
5,N006,A0507,work_or_schedule,0.98,The patient missed the appointment due to an u...,Reschedule the patient's appointment to an ava...,valid,"{""root_cause"":""work_or_schedule"",""confidence"":..."
6,N007,A0333,forgot_or_no_reminder,0.95,Patient forgot the appointment because no SMS ...,Reschedule the patient's appointment and verif...,valid,"{\n ""root_cause"": ""forgot_or_no_reminder"",\n ..."
7,N008,A0222,distance_or_transparent,0.95,Patient missed the appointment due to a city b...,Reschedule the appointment and offer alternati...,invalid,"{\n ""root_cause"": ""distance_or_transparent"",\..."
8,N009,A0977,other,0.90,Patient could not be reached after three attem...,Try contacting the patient at a later time or ...,valid,"{\n ""root_cause"": ""other"",\n ""confidence"": 0..."
9,N010,A0596,work_or_schedule,0.95,The patient missed the appointment due to a fa...,Reschedule the patient's appointment for an av...,valid,"{\n ""root_cause"": ""work_or_schedule"",\n ""con..."


##Task 4.3 — Break it on purpose

In [ ]:
#Create two invalid dictionaries

bad_replies = [
    (
        "Confidence above limit",
        {
            "root_cause": "other",
            "confidence": 1.7,
            "summary": "Patient missed an appointment.",
            "recommended_action": "Call the patient again."
        }
    ),
    (
        "Missing recommended_action",
        {
            "root_cause": "other",
            "confidence": 0.5,
            "summary": "Patient missed an appointment."
        }
    )
]

for label, reply in bad_replies:
  try:
    FollowupInsight(**reply)
    print(label, "unexpectedly passed")
  except ValidationError as error:
    print(f"{label}:\n{error}\n")

Confidence above limit:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal

Missing recommended_action:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...missed an appointment.'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing



In [ ]:
#For notes N007 (Hinglish) and N009 (ambiguous)

#V1

def classify_prompt(prompt_text):
  global last_raw_response
  response = client.models.generate_content(
      model = MODEL_NAME,
      contents = prompt_text,
      config = types.GenerateContentConfig(
          system_instruction=SYSTEM_TEXT,
          temperature=0,
          max_output_tokens=1000,
          response_mime_type="application/json",
          response_schema=SCHEMA,
          thinking_config=types.ThinkingConfig(thinking_level="minimal")
      )
  )
  last_raw_response = response.text
  return json.loads(last_raw_response)

n007_text = notes.loc[notes["note_id"].eq("N007"), "note_text"].iloc[0]
n009_text = notes.loc[notes["note_id"].eq("N009"), "note_text"].iloc[0]

v1_n007 = classify_prompt(f"Classify this note:\n{n007_text}")
time.sleep(13)
v1_n009 = classify_prompt(f"Classify this note:\n{n009_text}")

In [ ]:
#V2

COMPARE_NOTE_IDS = ["N007", "N009"]

#Read the note text for the two comparisons.
source_notes = pd.read_csv("/content/followup_notes.csv", dtype={"note_id":str})
source_notes = source_notes.set_index("note_id")

#Read the V2 results already produced by Task 4.2.
task42_results = pd.read_csv("notes_analysed.csv", dtype={"note_id": str})
task42_results = task42_results.set_index("note_id")

v1_results = {}
v1_api_calls = 0
v1_errors = {}

for index, note_id in enumerate(COMPARE_NOTE_IDS):
  if index > 0:
    time.sleep(13) #pause between the two V1 API calls.

  note_text = str(source_notes.loc[note_id, "note_text"]).strip()
  prompt_v1 = f"Classify this note:\n\n{note_text}"

  v1_api_calls += 1

  try:
    response = client.models.generate_content(
        model = MODEL_NAME,
        contents = prompt_v1,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config = types.ThinkingConfig(
                thinking_level="minimal"
            ),
        ),
    )

    raw_v1 = response.text or ""
    data_v1 = json.loads(raw_v1)

    if isinstance(data_v1, dict):
      v1_results[note_id] = data_v1.get("root_cause", "")
    else:
      v1_results[note_id] = ""
      v1_errors[note_id] = "V1 response was not a JSON object."

  except Exception as exc:
    v1_results[note_id] = ""
    v1_errors[note_id] = str(exc)
    print(f"{note_id} V1 call failed: {exc}")

#Compare each V1 result with the saved V2 Task 4.2 cause.

comparison_rows = []

for note_id in COMPARE_NOTE_IDS:
  v2_cause = task42_results.loc[note_id, "root_cause"]
  if pd.isna(v2_cause):
    v2_cause = ""
  else:
    v2_cause = str(v2_cause).strip()

  for version, cause in [
      ("V1", v1_results.get(note_id,"")),
      ("V2", v2_cause),
  ]:
      cause: ""if pd.isna(cause) else str(cause).strip()

      comparison_rows.append({
          "note_id": note_id,
          "prompt_version": version,
          "root_cause": cause,
          "allowed_root_cause": cause in ALLOWED_ROOT_CAUSES,
      })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)

#Deploy V2 only if both saved V2 causes are allowed.
v2_rows = comparison_df.loc[comparison_df["prompt_version"] == "V2"]
deploy_v2 = (
    len(v2_rows) == 2
    and bool(v2_rows["allowed_root_cause"].all())
)

if deploy_v2:
  print("Recommendation: deploy V2; both causes are in the allowed list.")
else:
  print("Recommendation: do not deply V2 yet; check the V2 causes.")

if v1_errors:
   print("\nV1 errors:", v1_errors)

#Enter the actual number of Task 4.1 API Calls, including repeated N001 calls.
task41_api_calls = int(input(
    "Enter the total Task 4.1 APL Calls made for N001, including reruns: "
))

task42_api_calls = 9 #From Task 4.2 output.

total_api_calls = task41_api_calls + task42_api_calls + v1_api_calls

print(f"\nTask 4.1 API calls: {task41_api_calls}")
print(f"\nTask 4.2 API calls: {task42_api_calls}")
print(f"\nTask 4.3 V1 API calls: {v1_api_calls}")
print(f"\nTotal API calls: {total_api_calls}")

,note_id,prompt_version,root_cause,allowed_root_cause
0,N007,V1,Forgot appointment and lack of SMS reminder,False
1,N007,V2,forgot_or_no_reminder,True
2,N009,V1,Unreachable by phone,False
3,N009,V2,other,True


Recommendation: deploy V2; both causes are in the allowed list.
Enter the total Task 4.1 APL Calls made for N001, including reruns: 2

Task 4.1 API calls: 2

Task 4.2 API calls: 9

Task 4.3 V1 API calls: 2

Total API calls: 13


##Task 4.4 — Recommendation and GitHub

In [ ]:
#Task 4.4(a). Merge the ten analyzed notes to the appointment reminder field

notes_df = pd.read_csv("/content/notes_analysed.csv")
appointment_df = pd.read_csv("/content/appointments_clean.csv")

In [ ]:
merged = notes_df.merge(
    appointment_df[["appointment_id", "reminder_sent"]],
    on="appointment_id",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", merged.shape)
print(pd.crosstab(merged["root_cause"], merged["reminder_sent"]))

Merged shape: (10, 9)
reminder_sent            No  Yes
root_cause                      
cost_or_fee               0    1
distance_or_transparent   1    1
forgot_or_no_reminder     2    0
illness_or_emergency      0    1
other                     1    1
work_or_schedule          1    1
